In [1]:
import json
from tqdm.auto import tqdm

TRAIN_FILE = "/kaggle/input/datasets/kishorer2k4/maven-translated/train.jsonl"

HINDI_FILE = "/kaggle/input/datasets/kishorer2k4/maven-translated/maven_hindi.json"
MALAYALAM_FILE = "/kaggle/input/datasets/kishorer2k4/maven-translated/maven_malayalam.json"
FRENCH_FILE = "/kaggle/input/datasets/kishorer2k4/maven-translated/maven_french.json"

with open(HINDI_FILE, "r", encoding="utf-8") as f:
    hindi = json.load(f)

with open(MALAYALAM_FILE, "r", encoding="utf-8") as f:
    malayalam = json.load(f)

with open(FRENCH_FILE, "r", encoding="utf-8") as f:
    french = json.load(f)

print(len(hindi), len(malayalam), len(french))

32431 32431 32431


In [2]:
import json
from tqdm.auto import tqdm

merged = []

idx = 0

with open(TRAIN_FILE, "r", encoding="utf-8") as f:

    for line in tqdm(f):

        doc = json.loads(line)

        for sent_id, sent in enumerate(doc["content"]):

            merged.append({
                "doc_id": doc["id"],
                "sent_id": sent_id,

                "english": sent["sentence"],
                "hindi": hindi[idx],
                "malayalam": malayalam[idx],
                "french": french[idx]
            })

            idx += 1

print("Merged:", len(merged))

0it [00:00, ?it/s]

Merged: 32431


In [3]:
OUTPUT = "/kaggle/working/maven_multilingual.json"

with open(
    OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        merged,
        f,
        ensure_ascii=False
    )

print("Saved:", OUTPUT)

Saved: /kaggle/working/maven_multilingual.json


In [4]:
enhanced = []

idx = 0

with open(TRAIN_FILE, "r", encoding="utf-8") as f:

    for line in tqdm(f):

        doc = json.loads(line)

        for sent_id, sent in enumerate(doc["content"]):

            enhanced.append({
                "doc_id": doc["id"],
                "title": doc["title"],
                "sent_id": sent_id,

                "english": sent["sentence"],
                "hindi": hindi[idx],
                "malayalam": malayalam[idx],
                "french": french[idx],

                "events": doc["events"]
            })

            idx += 1

print(len(enhanced))

0it [00:00, ?it/s]

32431


In [5]:
with open(
    "/kaggle/working/maven_multilingual_full.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        enhanced,
        f,
        ensure_ascii=False
    )

In [6]:
!pip install -q transformers sentence-transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.7 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [7]:
import transformers
import sentence_transformers
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

print("Transformers:", transformers.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Using:", device)

# XLM-R
tokenizer = AutoTokenizer.from_pretrained(
    "xlm-roberta-base"
)

encoder = AutoModel.from_pretrained(
    "xlm-roberta-base"
).to(device)

# LaBSE
labse = SentenceTransformer(
    "sentence-transformers/LaBSE",
    device=device
)

print("Tokenizer loaded")
print("Encoder loaded")
print("LaBSE loaded")

Transformers: 5.0.0
SentenceTransformers: 5.4.0
Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
Using: cuda


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Tokenizer loaded
Encoder loaded
LaBSE loaded


In [8]:
import json

MULTI_FILE = "/kaggle/working/maven_multilingual.json"

with open(MULTI_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Samples:", len(data))
print(data[0].keys())

Samples: 32431
dict_keys(['doc_id', 'sent_id', 'english', 'hindi', 'malayalam', 'french'])


In [9]:
from torch.utils.data import Dataset

class AlignmentDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        item = self.data[idx]

        return {
            "en": item["english"],
            "hi": item["hindi"]
        }

dataset = AlignmentDataset(data)

print(len(dataset))

32431


In [10]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

In [11]:
import torch.nn as nn

class ProjectionHead(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, 256)
        self.act = nn.GELU()

    def forward(self, x):

        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)

        return x

projection = ProjectionHead().to(device)

print("Projection loaded")

Projection loaded


In [12]:
import torch

def mean_pool(last_hidden_state, attention_mask):

    mask = attention_mask.unsqueeze(-1).float()

    summed = (last_hidden_state * mask).sum(1)

    counts = mask.sum(1)

    return summed / counts

In [13]:
import torch.nn.functional as F

def encode_texts(texts):

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    outputs = encoder(**inputs)

    pooled = mean_pool(
        outputs.last_hidden_state,
        inputs["attention_mask"]
    )

    projected = projection(pooled)

    projected = F.normalize(
        projected,
        p=2,
        dim=1
    )

    return projected

## Dataset Format

In [14]:
import json

with open("/kaggle/working/maven_multilingual_full.json") as f:
    data = json.load(f)

sample = data[0]

print("Keys:", sample.keys())
print("\nEvents Example:\n")
print(sample["events"][0])

Keys: dict_keys(['doc_id', 'title', 'sent_id', 'english', 'hindi', 'malayalam', 'french', 'events'])

Events Example:

{'id': '40b3b20bc2eeb6b163538b82c1379ead', 'type': 'Know', 'type_id': 1, 'mention': [{'trigger_word': 'observed', 'sent_id': 5, 'offset': [12, 13], 'id': '7fcf445a679aa13511278d321a908bd2'}]}


## Event classes

In [15]:
event_types = set()

for item in data:
    for event in item["events"]:
        event_types.add(event["type"])

print("Classes:", len(event_types))
print(sorted(list(event_types))[:20])


Classes: 168
['Achieve', 'Action', 'Adducing', 'Agree_or_refuse_to_act', 'Aiming', 'Arranging', 'Arrest', 'Arriving', 'Assistance', 'Attack', 'Award', 'Bearing_arms', 'Becoming', 'Becoming_a_member', 'Being_in_operation', 'Besieging', 'Bodily_harm', 'Body_movement', 'Breathing', 'Bringing']


## paired event

In [16]:
from collections import defaultdict

sentence_labels = []

for row in data:

    labels = set()

    current_sent = row["sent_id"]

    for event in row["events"]:

        for mention in event["mention"]:

            if mention["sent_id"] == current_sent:
                labels.add(event["type"])

    sentence_labels.append(list(labels))

print("Examples")

for i in range(5):
    print(sentence_labels[i])

Examples
['Catastrophe', 'Presence']
['Know', 'Catastrophe']
['Catastrophe', 'Influence', 'Motion', 'Causation', 'Destroying', 'Damaging']
['Placing', 'Catastrophe']
['Destroying', 'Catastrophe', 'Protest', 'Causation']


In [17]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

Y = mlb.fit_transform(sentence_labels)

print(Y.shape)
print(len(mlb.classes_))

(32431, 168)
168


In [18]:
for i in range(len(data)):
    data[i]["label"] = Y[i]

In [19]:
from torch.utils.data import Dataset

class EventDataset(Dataset):

    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):

        row = self.rows[idx]

        return {
            "text": row["english"],
            "labels": row["label"]
        }

In [20]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    data,
    test_size=0.1,
    random_state=42
)

train_ds = EventDataset(train_data)
val_ds = EventDataset(val_data)

print(len(train_ds))
print(len(val_ds))

29187
3244


In [21]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=16
)

## Import XLMR from torch.nn module

In [22]:
import torch
import torch.nn as nn

class XLMREventClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = encoder

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            768,
            168
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:,0]

        cls = self.dropout(cls)

        logits = self.classifier(cls)

        return logits

model = XLMREventClassifier().to(device)

In [23]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

## Training 

In [25]:
from tqdm.auto import tqdm
import torch

EPOCHS = 3

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    loop = tqdm(train_loader)

    for batch in loop:

        texts = batch["text"]

        labels = batch["labels"].float().to(device)

        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        enc = {
            k:v.to(device)
            for k,v in enc.items()
        }

        optimizer.zero_grad()

        logits = model(**enc)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        loop.set_description(
            f"Epoch {epoch+1}"
        )

        loop.set_postfix(
            loss=loss.item()
        )

    print(
        f"\nEpoch {epoch+1} Avg Loss:",
        total_loss / len(train_loader)
    )

  0%|          | 0/1825 [00:00<?, ?it/s]


Epoch 1 Avg Loss: 0.058413223028182984


  0%|          | 0/1825 [00:00<?, ?it/s]


Epoch 2 Avg Loss: 0.04395082424764764


  0%|          | 0/1825 [00:00<?, ?it/s]


Epoch 3 Avg Loss: 0.034357124317998755


## F1 Result

In [49]:
from sklearn.metrics import f1_score
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(val_loader):

        texts = batch["text"]

        labels = np.array(batch["labels"])

        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        enc = {
            k:v.to(device)
            for k,v in enc.items()
        }

        logits = model(**enc)

        probs = torch.sigmoid(
            logits
        ).cpu().numpy()

        preds = (probs > 0.5).astype(int)

        all_preds.append(preds)
        all_labels.append(labels)

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

macro_f1 = f1_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

micro_f1 = f1_score(
    all_labels,
    all_preds,
    average="micro",
    zero_division=0
)

print("Macro F1:", macro_f1)
print("Micro F1:", micro_f1)

  0%|          | 0/203 [00:00<?, ?it/s]

/tmp/ipykernel_58/2762255873.py:15: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  labels = np.array(batch["labels"])


Macro F1: 0.7768642772209954
Micro F1: 0.6653056194012095


In [27]:
torch.save(
    model.state_dict(),
    "/kaggle/working/xlmr_event_baseline.pt"
)

In [28]:
from collections import Counter
import numpy as np

freq = Y.sum(axis=0)

weights = 1 / np.sqrt(freq + 1)

weights = weights / weights.mean()

class_weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

print(class_weights.shape)

torch.Size([168])


In [29]:

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=class_weights
)

In [30]:
import random
from torch.utils.data import Dataset

class MultiLingualDataset(Dataset):

    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):

        row = self.rows[idx]

        text = random.choice([
            row["english"],
            row["hindi"],
            row["malayalam"],
            row["french"]
        ])

        return {
            "text": text,
            "labels": row["label"]
        }

In [31]:
train_ds = MultiLingualDataset(train_data)

In [32]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    "xlm-roberta-base"
)

encoder = AutoModel.from_pretrained(
    "xlm-roberta-base"
)

class XLMREventClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = encoder

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            768,
            168
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:,0]

        cls = self.dropout(cls)

        return self.classifier(cls)

model = XLMREventClassifier().to(device)

model.load_state_dict(
    torch.load(
        "/kaggle/working/xlmr_event_baseline.pt",
        map_location=device
    )
)

model.eval()

print("Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded


In [33]:
event_names = list(mlb.classes_)

print(event_names[:10])
print(len(event_names))

['Achieve', 'Action', 'Adducing', 'Agree_or_refuse_to_act', 'Aiming', 'Arranging', 'Arrest', 'Arriving', 'Assistance', 'Attack']
168


In [34]:
import torch

def predict_events(
    text,
    threshold=0.5
):

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    enc = {
        k:v.to(device)
        for k,v in enc.items()
    }

    with torch.no_grad():

        logits = model(**enc)

        probs = torch.sigmoid(
            logits
        ).cpu().numpy()[0]

    predictions = []

    for i,p in enumerate(probs):

        if p >= threshold:

            predictions.append(
                (
                    event_names[i],
                    float(p)
                )
            )

    predictions.sort(
        key=lambda x:x[1],
        reverse=True
    )

    return predictions

In [35]:
def predict_topk(text, k=5):

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    enc = {
        key:v.to(device)
        for key,v in enc.items()
    }

    with torch.no_grad():

        logits = model(**enc)

        probs = torch.sigmoid(logits)[0]

    topk = torch.topk(probs, k)

    results = []

    for score, idx in zip(
        topk.values,
        topk.indices
    ):
        results.append(
            (
                mlb.classes_[idx.item()],
                float(score)
            )
        )

    return results

## Prediction result store (after prediction of event classes)

In [36]:
event_db = []

for row in data:

    text = row["english"]

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    enc = {
        k:v.to(device)
        for k,v in enc.items()
    }

    with torch.no_grad():

        logits = model(**enc)

        probs = torch.sigmoid(logits)[0]

    topk = torch.topk(probs, 5)

    predictions = []

    for score, idx in zip(
        topk.values,
        topk.indices
    ):

        predictions.append({
            "event": mlb.classes_[idx.item()],
            "score": float(score)
        })

    event_db.append({
        "english": row["english"],
        "hindi": row["hindi"],
        "malayalam": row["malayalam"],
        "french": row["french"],
        "predictions": predictions
    })

print("Done:", len(event_db))

Done: 32431


In [37]:
import json

with open(
    "/kaggle/working/event_database.json",
    "w"
) as f:

    json.dump(
        event_db,
        f,
        ensure_ascii=False
    )

print("Saved")

Saved


## Usage

## Fetch news online

In [38]:
!pip install -q gnews newspaper4k lxml_html_clean

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 7.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 64.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.3 MB/s eta 0:00:00


In [39]:
from gnews import GNews

def fetch_news(topic, lang="en", max_results=10):
    """Fetch news articles for a topic in a given language."""

    google_news = GNews(
        language=lang,
        max_results=max_results,
        period="7d"
    )

    articles = google_news.get_news(topic)

    results = []

    for article in articles:

        results.append({
            "title": article.get("title", ""),
            "description": article.get("description", ""),
            "published": article.get("published date", ""),
            "publisher": article.get("publisher", {}).get("title", ""),
            "url": article.get("url", "")
        })

    return results

print("fetch_news ready")

fetch_news ready


In [40]:
LANG_MAP = {
    "english":   "en",
    "hindi":     "hi",
    "malayalam": "ml",
    "french":    "fr"
}

def fetch_multilingual_news(topic, max_per_lang=5):
    """Fetch news for a topic in all 4 languages."""

    all_news = {}

    for lang_name, lang_code in LANG_MAP.items():

        print(f"Fetching {lang_name} news...")

        try:
            articles = fetch_news(
                topic,
                lang=lang_code,
                max_results=max_per_lang
            )
            all_news[lang_name] = articles
            print(f"  Found {len(articles)} articles")

        except Exception as e:
            print(f"  Error: {e}")
            all_news[lang_name] = []

    return all_news

print("fetch_multilingual_news ready")

fetch_multilingual_news ready


In [41]:
def detect_events_in_news(news_dict, top_k=5):
    """Run event detection on fetched multilingual news."""

    results = []

    for lang_name, articles in news_dict.items():

        for article in articles:

            text = article["title"]

            if not text.strip():
                continue

            preds = predict_topk(text, k=top_k)

            results.append({
                "language": lang_name,
                "title": text,
                "publisher": article["publisher"],
                "published": article["published"],
                "url": article["url"],
                "events": [
                    {"type": e, "score": round(s, 3)}
                    for e, s in preds
                ]
            })

    return results

print("detect_events_in_news ready")

detect_events_in_news ready


In [42]:
def display_results(results):
    """Pretty-print event detection results."""

    current_lang = None

    for r in results:

        if r["language"] != current_lang:
            current_lang = r["language"]
            print("" + "#" * 80)
            print(f"  {current_lang.upper()}")
            print("#" * 80)

        print("" + "-" * 60)
        print(f"  {r["title"]}")
        print(f"  Source: {r["publisher"]} | {r["published"]}")
        print(f"  Events:")

        for event in r["events"]:
            bar = " " * int(event["score"] * 20)
            print(f"{event["type"]:30s} "f"{event["score"]:.3f} {bar}")

print("display_results ready")

display_results ready


In [43]:
# Install Hugging Face libraries for local LLM summarization
!pip install -q transformers accelerate


## Local LLM summariser (Qwen)

In [51]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

print("Loading local multilingual LLM (Qwen/Qwen2.5-1.5B-Instruct)...")
print("This model runs entirely on Kaggle GPU — no API keys, no internet needed.")

# Determine device and dtype automatically
if torch.cuda.is_available():
    _device_map = "auto"
    _torch_dtype = torch.float16
    print("Using CUDA GPU for inference.")
else:
    _device_map = None
    _torch_dtype = torch.float32
    print("Using CPU for inference (slower).")

try:
    _qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
    _qwen_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        torch_dtype=_torch_dtype,
        device_map=_device_map
    )

    _summariser_pipe = pipeline(
        "text-generation",
        model=_qwen_model,
        tokenizer=_qwen_tokenizer
    )
    _local_llm_ready = True
    print("Qwen-2.5-1.5B-Instruct loaded successfully!")
except Exception as e:
    _local_llm_ready = False
    print(f"Error loading local model: {e}")
    print("Falling back to structured text summaries.")


def summarise_news(articles, event_type):
    """Summarise multilingual news locally using Qwen2.5-1.5B-Instruct."""

    # Group headlines by language
    by_lang = {}
    for a in articles:
        lang = a["language"]
        if lang not in by_lang:
            by_lang[lang] = []
        by_lang[lang].append(a)

    text_block = ""
    for lang, items in by_lang.items():
        text_block += f"\n--- {lang.upper()} ---\n"
        for a in items:
            text_block += f"- {a['title']} (Source: {a['publisher']})\n"

    # Structured fallback if LLM is unavailable
    def get_structured_fallback():
        lines = [f"Event: {event_type}"]
        lines.append(f"Matched {len(articles)} articles across {len(by_lang)} languages\n")
        for lang, items in by_lang.items():
            lines.append(f"  {lang.upper()}:")
            for a in items:
                lines.append(f"    \u2022 {a['title']} ({a['publisher']})")
        return "\n".join(lines)

    if not _local_llm_ready:
        return get_structured_fallback()

    prompt = (
        f"You are a multilingual news analyst.\n\n"
        f"The user searched for the event type: {event_type}\n\n"
        f"Below are news headlines in various languages: {', '.join(by_lang.keys())}.\n"
        f"All these articles were identified by an event detector as matching \\\"{event_type}\\\"\n\n"
        f"Headlines:\n{text_block}\n\n"
        f"Please provide a concise 3-4 sentence summary of what is happening globally regarding this event type, "
        f"grouping key themes across languages and noting any regional patterns. "
        f"Do not repeat the prompt. Keep it informative, objective, and neutral."
        f"Give the summary in french, malayalam and hindi languages"
        
    )

    messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise and write a high-quality summary."},
        {"role": "user", "content": prompt}
    ]

    try:
        outputs = _summariser_pipe(
            messages,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
        summary = outputs[0]["generated_text"][-1]["content"].strip()
        return summary
    except Exception as e:
        print(f"Error during summarization: {e}")
        return get_structured_fallback()


print("Local summariser function ready")


06/05/2026 05:43:12 AM - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/05/2026 05:43:12 AM - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
06/05/2026 05:43:12 AM - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


Loading local multilingual LLM (Qwen/Qwen2.5-1.5B-Instruct)...
This model runs entirely on Kaggle GPU — no API keys, no internet needed.
Using CUDA GPU for inference.


06/05/2026 05:43:12 AM - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"
06/05/2026 05:43:12 AM - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
06/05/2026 05:43:12 AM - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
06/05/2026 05:43:13 AM - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
06/05/2026 05:43:13 AM - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/05/2026 05:43:13 AM - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

06/05/2026 05:43:15 AM - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
06/05/2026 05:43:15 AM - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/generation_config.json "HTTP/1.1 200 OK"


Qwen-2.5-1.5B-Instruct loaded successfully!
Local summariser function ready


In [52]:
# Map event types to Google News search keywords
EVENT_TO_QUERY = {
    "Catastrophe": "earthquake OR tsunami OR flood OR disaster",
    "Attack": "attack OR bombing OR assault",
    "Military_operation": "military operation OR airstrike OR war",
    "Protest": "protest OR demonstration OR rally",
    "Killing": "killing OR murder OR assassination",
    "Terrorism": "terrorism OR terrorist attack",
    "Destroying": "destruction OR destroyed OR demolition",
    "Death": "death OR died OR killed",
    "Hostile_encounter": "conflict OR clash OR battle",
    "Change_of_leadership": "election OR prime minister OR president elected",
    "Legal_rulings": "court ruling OR verdict OR sentenced",
    "Arrest": "arrested OR arrest OR detained",
    "Fire": "fire OR wildfire OR blaze",
}

def get_search_query(event_type):
    """Get a search query for an event type. Falls back to the event name itself."""

    return EVENT_TO_QUERY.get(
        event_type,
        event_type.replace("_", " ")
    )

print("Event-to-query mapping ready")
print(f"Pre-mapped events: {len(EVENT_TO_QUERY)}")
print("Unmapped events will use event name as search query")

Event-to-query mapping ready
Pre-mapped events: 13
Unmapped events will use event name as search query


In [53]:
def mode1_event_to_news(
    event_type,
    max_per_lang=5,
    min_score=0.3
):
    """
    Mode 1: Event Type → Multilingual News + Summary

    1. Convert event type to search query
    2. Fetch news in all 4 languages
    3. Run event detection on each headline
    4. Filter: keep only articles where the model detects the target event
    5. Summarise with local LLM
    """

    query = get_search_query(event_type)
    print(f"\nSearching for: \"{query}\"  (event: {event_type})")
    print("=" * 80)

    # Fetch
    news = fetch_multilingual_news(
        query,
        max_per_lang=max_per_lang
    )

    # Detect + Filter
    all_results = detect_events_in_news(news)

    filtered = []

    for r in all_results:

        for e in r["events"]:

            if (
                e["type"].lower() == event_type.lower()
                and e["score"] >= min_score
            ):
                r["matched_score"] = e["score"]
                filtered.append(r)
                break

    filtered.sort(
        key=lambda x: x["matched_score"],
        reverse=True
    )

    print(f"\nFetched {len(all_results)} articles → "
          f"{len(filtered)} match \"{event_type}\" (score ≥ {min_score})")

    if not filtered:
        print("No matching articles found.")
        return

    # Display
    print("\n" + "─" * 80)
    print("  MATCHING ARTICLES")
    print("─" * 80)

    for r in filtered:

        lang_tag = r["language"][:2].upper()

        print(
            f"  [{lang_tag}] {r["title"]}\n"
            f"       Source: {r["publisher"]} | {r["published"]}\n"
            f"       {event_type} score: {r["matched_score"]:.3f}"
        )

    # Summarise
    print("\n" + "─" * 80)
    print("  LLM SUMMARY")
    print("─" * 80 + "\n")

    summary = summarise_news(filtered, event_type)

    print(summary)

    return filtered

print("Mode 1 ready: Event → News + Summary")

Mode 1 ready: Event → News + Summary


In [54]:
def mode2_news_to_events(
    text,
    top_k=10,
    threshold=0.1
):
    """
    Mode 2: News Text (any language) → Event Classes

    Paste any headline or sentence in any language.
    The model predicts event types with confidence scores.
    """

    print(f"\nInput: {text}")
    print("=" * 80)

    # Threshold-based
    threshold_preds = predict_events(
        text,
        threshold=threshold
    )

    # Top-K
    topk_preds = predict_topk(
        text,
        k=top_k
    )

    print(f"\nDetected Events (score ≥ {threshold}):\n")

    if threshold_preds:

        for event, score in threshold_preds:

            bar = "█" * int(score * 30)
            marker = " ◄ HIGH" if score >= 0.5 else ""
            print(f"  {event:30s} {score:.3f} {bar}{marker}")

    else:
        print("  No events above threshold.\n")
        print(f"  Top-{top_k} predictions:\n")

        for event, score in topk_preds:

            bar = "█" * int(score * 30)
            print(f"  {event:30s} {score:.3f} {bar}")

    return threshold_preds or topk_preds

print("Mode 2 ready: News → Events")

Mode 2 ready: News → Events


## User side 

In [ ]:
while True:

    print("\n" + "=" * 80)
    print("  CLED — Cross-Lingual Event Detection")
    print("=" * 80)
    print("\n  [1] Event Type → Fetch multilingual news + LLM summary")
    print("  [2] Paste news text → Detect event classes")
    print("  [3] Exit\n")

    choice = input("Select mode (1/2/3): ").strip()

    if choice == "1":

        print(f"\nAvailable event types (sample): ")
        print(", ".join(event_names[:20]))
        print(f"... ({len(event_names)} total)\n")

        event_type = input("Enter event type: ").strip()

        if not event_type:
            continue

        mode1_event_to_news(event_type)

    elif choice == "2":

        text = input("\nPaste news headline (any language): ").strip()

        if not text:
            continue

        mode2_news_to_events(text)

    elif choice == "3":
        print("\nGoodbye!")
        break

    else:
        print("Invalid choice. Enter 1, 2, or 3.")


  CLED — Cross-Lingual Event Detection

  [1] Event Type → Fetch multilingual news + LLM summary
  [2] Paste news text → Detect event classes
  [3] Exit



Select mode (1/2/3):  1



Available event types (sample): 
Achieve, Action, Adducing, Agree_or_refuse_to_act, Aiming, Arranging, Arrest, Arriving, Assistance, Attack, Award, Bearing_arms, Becoming, Becoming_a_member, Being_in_operation, Besieging, Bodily_harm, Body_movement, Breathing, Bringing
... (168 total)



Enter event type:  attack 



Searching for: "attack"  (event: attack)
Fetching english news...
  Found 5 articles
Fetching hindi news...
  Found 5 articles
Fetching malayalam news...
  Found 5 articles
Fetching french news...


...


Input: கோடை விடுமுறைக்குப் பிறகு, தமிழகம் மற்றும் புதுச்சேரியில் உள்ள பள்ளிகள் நாளை மீண்டும் திறக்கப்பட உள்ளன. மாணவர்களை உற்சாகத்துடன் வரவேற்குமாறு பள்ளிக் கல்வித் துறை அறிவுறுத்தியுள்ளது.

Detected Events (score ≥ 0.1):

  Openness                       0.229 ██████
  Request                        0.144 ████
  Process_end                    0.108 ███

  CLED — Cross-Lingual Event Detection

  [1] Event Type → Fetch multilingual news + LLM summary
  [2] Paste news text → Detect event classes
  [3] Exit



Select mode (1/2/3):   les Cognaçais du Team Grip Attack dans la cour des grands à Spa


Invalid choice. Enter 1, 2, or 3.

  CLED — Cross-Lingual Event Detection

  [1] Event Type → Fetch multilingual news + LLM summary
  [2] Paste news text → Detect event classes
  [3] Exit



Select mode (1/2/3):  2

Paste news headline (any language):   les Cognaçais du Team Grip Attack dans la cour des grands à Spa



Input: les Cognaçais du Team Grip Attack dans la cour des grands à Spa

Detected Events (score ≥ 0.1):

  Attack                         0.843 █████████████████████████ ◄ HIGH

  CLED — Cross-Lingual Event Detection

  [1] Event Type → Fetch multilingual news + LLM summary
  [2] Paste news text → Detect event classes
  [3] Exit

